# **Employee Data Profiling Overview**

This notebook performs data profiling on the Employee dataset to evaluate structure, quality, and consistency before further processing.

## Objectives

* Validate schema (columns, data types)
* Check missing values and duplicates
* Analyze data distribution (age, salary, etc.)
* Detect anomalies and inconsistencies

## Scope

* Row & column count
* Data types validation
* Null value analysis
* Distinct count (cardinality)
* Summary stats (min, max, mean, stddev)
* Basic outlier checks

## Outcome

Provides insights into data quality and readiness for cleaning, transformation, and analysis.

**Status:** Data profiling initiated...
**Author:** Ritik
**Env:** Development / Testing


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

print("Required libraries imported successfully. :) ")

Required libraries imported successfully. :) 


In [7]:
spark = (
    SparkSession.builder
    .appName("EmployeesDataProfiling")
    .getOrCreate()
)

print("SparkSession Build Successfully :) ")

SparkSession Build Successfully :) 


In [8]:
try :
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("/content/the_data_echo/row_data/raw_employees.csv")
    )
    print(f"Data Loaded successfully row_count {df.count()}  :) ")

except Exception as e :
    print(f"Error Occurred : {e}")

Data Loaded successfully row_count 100  :) 


In [9]:
df.sample(fraction=0.1, withReplacement=False).show(truncate=False)

+-----------+------------+---------+--------------+--------------------------------+--------------+--------------------+----------------+--------+---------------------+------------+------------------+--------------+-----------------+-------------------+---------+------------------+----------+
|employee_id|first_name  |last_name|full_name     |email                           |phone         |job_title           |department      |store_id|store_name           |store_city  |hire_date         |years_employed|annual_salary_usd|commission_rate_pct|is_active|performance_rating|manager_id|
+-----------+------------+---------+--------------+--------------------------------+--------------+--------------------+----------------+--------+---------------------+------------+------------------+--------------+-----------------+-------------------+---------+------------------+----------+
|4003       |Janet       |  Torres |Janet Torres  |janet_torres454@aol.com         |430.836.4218  |Regional Manager   

## **Employee ID data profiling**

In [10]:
# Null check
df.select("employee_id").filter(F.col("employee_id").isNull()).count()

0

In [11]:
# Uniqueness (duplicate)
df.groupBy("employee_id").count().filter("count > 1").show()

+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+



In [12]:
# Data type validation
df.select("employee_id").dtypes

[('employee_id', 'int')]

In [13]:
# Range check
df.select("employee_id").filter(F.col("employee_id") < 1).show()

+-----------+
|employee_id|
+-----------+
+-----------+



In [14]:
df.select("employee_id").sample(fraction=0.1, withReplacement=False).show()

+-----------+
|employee_id|
+-----------+
|       4019|
|       4030|
|       4033|
|       4035|
|       4040|
|       4043|
|       4054|
|       4055|
|       4072|
|       4084|
|       4097|
|       4099|
+-----------+



## **Manager ID data Profiling**

In [15]:

df.select("first_name","manager_id").filter(F.col("manager_id").isNull()).show()

+----------+----------+
|first_name|manager_id|
+----------+----------+
|    George|      NULL|
|     Emily|      NULL|
|     Laura|      NULL|
|     James|      NULL|
|     Frank|      NULL|
|   Joshua |      NULL|
|     tyler|      NULL|
|  nicholas|      NULL|
|     Jacob|      NULL|
|   Deborah|      NULL|
|   Deborah|      NULL|
|     Joyce|      NULL|
|     Frank|      NULL|
|      Mark|      NULL|
|     Laura|      NULL|
|   CAROLYN|      NULL|
|     Diane|      NULL|
+----------+----------+



In [16]:
df.select("manager_id").filter(F.col("manager_id").isNull()).count()

17

In [17]:
df.select("manager_id").filter(F.col("manager_id").isNotNull()).count()

83

In [18]:
df.groupBy("manager_id").count().filter(F.col("manager_id").isNotNull()).filter("count > 1").show()

+----------+-----+
|manager_id|count|
+----------+-----+
|      4092|    2|
|      4061|    2|
|      4040|    2|
|      4032|    2|
|      4012|    2|
|      4037|    2|
|      4007|    2|
|      4023|    2|
|      4068|    2|
|      4020|    3|
|      4066|    2|
|      4010|    2|
|      4086|    2|
|      4055|    2|
|      4053|    2|
|      4098|    2|
|      4050|    2|
|      4003|    2|
|      4096|    2|
|      4021|    2|
+----------+-----+
only showing top 20 rows


In [19]:
df.select("manager_id").dtypes

[('manager_id', 'int')]

## **performance_rating Data Profiling**

In [20]:
df.select("performance_rating").dtypes

[('performance_rating', 'string')]

In [21]:
df.select("performance_rating").distinct().show()

+------------------+
|performance_rating|
+------------------+
|         Excellent|
|                 3|
|                 5|
|                 B|
|           Average|
|     Below Average|
|              Good|
|                 D|
|                 C|
|                 A|
|                 4|
|                 2|
|              NULL|
+------------------+



In [22]:
df.select("performance_rating").filter(F.col("performance_rating").isNotNull()).count()

94

In [23]:
df.groupBy("performance_rating").count().show()

+------------------+-----+
|performance_rating|count|
+------------------+-----+
|         Excellent|   13|
|                 3|    9|
|              NULL|    6|
|                 5|    7|
|                 B|    4|
|           Average|    4|
|     Below Average|    1|
|              Good|   10|
|                 D|   12|
|                 C|   11|
|                 A|    7|
|                 4|    4|
|                 2|   12|
+------------------+-----+



## **Employee Is active or not data profiling**

In [24]:
df.select("is_active").dtypes

[('is_active', 'string')]

In [25]:
df.select("is_active").filter(F.col("is_active").isNull()).count()

0

In [26]:
df.select("is_active").distinct().show()

+----------+
| is_active|
+----------+
|         Y|
|    Active|
|         N|
|Terminated|
|        No|
|       Yes|
|         1|
+----------+



In [27]:
df.groupBy("is_active").count().show()

+----------+-----+
| is_active|count|
+----------+-----+
|         Y|   19|
|    Active|   19|
|         N|   14|
|Terminated|   12|
|        No|   10|
|       Yes|   13|
|         1|   13|
+----------+-----+



## **Commission_rate_pct Data Profiling**

In [28]:
df.select("commission_rate_pct").dtypes

[('commission_rate_pct', 'double')]

In [29]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct").isNull()).count()

0

In [30]:
df.sample(fraction=0.1, withReplacement=False).select("commission_rate_pct").show()

+-------------------+
|commission_rate_pct|
+-------------------+
|                3.1|
|                7.6|
|                6.8|
|                7.1|
|                6.5|
|                1.2|
|                7.1|
|                5.1|
|                1.8|
|                6.0|
|                7.6|
|                4.7|
+-------------------+



In [31]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct") < 0).show()

+-------------------+
|commission_rate_pct|
+-------------------+
+-------------------+



In [32]:
df.select("commission_rate_pct").filter(F.col("commission_rate_pct") > 100).show()

+-------------------+
|commission_rate_pct|
+-------------------+
+-------------------+



In [34]:
df.select(F.max("commission_rate_pct").alias("max_cmmission_rate")).show()

+------------------+
|max_cmmission_rate|
+------------------+
|               7.9|
+------------------+



In [37]:
df.select(F.min("commission_rate_pct").alias("min_cmmission_rate")).show()

+------------------+
|min_cmmission_rate|
+------------------+
|               1.1|
+------------------+



In [41]:
df.select(F.mode("commission_rate_pct").alias("mode_commission_rate")).show()

+--------------------+
|mode_commission_rate|
+--------------------+
|                 2.6|
+--------------------+



In [39]:
df.select("commission_rate_pct").summary().show()

+-------+-------------------+
|summary|commission_rate_pct|
+-------+-------------------+
|  count|                100|
|   mean|              4.326|
| stddev| 1.9599845392092743|
|    min|                1.1|
|    25%|                2.6|
|    50%|                4.2|
|    75%|                5.9|
|    max|                7.9|
+-------+-------------------+



## **Employee annual_salary_usd Data Profiling**

In [48]:
df.select("annual_salary_usd").show(10)

+-----------------+
|annual_salary_usd|
+-----------------+
|            75248|
|            84780|
|           116056|
|            93447|
|            92114|
|            76680|
|            40962|
|            63122|
|            36973|
|           102209|
+-----------------+
only showing top 10 rows


In [49]:
df.select("annual_salary_usd").dtypes

[('annual_salary_usd', 'int')]

In [50]:
df.select("annual_salary_usd").filter(F.col("annual_salary_usd").isNull()).count()

0

In [51]:
df.select("annual_salary_usd").filter(F.col("annual_salary_usd") < 0).count()

0

In [58]:
df.select("full_name", "department", "job_title", "annual_salary_usd")\
.agg(F.mean(F.col("annual_salary_usd")).alias("min_annual_salary_usd")).show(truncate=False)

+---------------------+
|min_annual_salary_usd|
+---------------------+
|75849.64             |
+---------------------+



In [67]:
df.groupBy("department")\
  .agg(F.round(F.mean(F.col("annual_salary_usd"))).alias("mean_annual_salary_usd"))\
  .orderBy(F.col("mean_annual_salary_usd").desc()).show()

+----------------+----------------------+
|      department|mean_annual_salary_usd|
+----------------+----------------------+
|      Operations|               82373.0|
|Customer Service|               74699.0|
|           Sales|               74171.0|
|      Management|               72644.0|
+----------------+----------------------+



In [43]:
df.show(truncate=False)

+-----------+----------+-----------+-----------------+------------------------------+--------------+----------------------+----------------+--------+---------------------+------------+------------------+--------------+-----------------+-------------------+---------+------------------+----------+
|employee_id|first_name|last_name  |full_name        |email                         |phone         |job_title             |department      |store_id|store_name           |store_city  |hire_date         |years_employed|annual_salary_usd|commission_rate_pct|is_active|performance_rating|manager_id|
+-----------+----------+-----------+-----------------+------------------------------+--------------+----------------------+----------------+--------+---------------------+------------+------------------+--------------+-----------------+-------------------+---------+------------------+----------+
|4001       |Frank     |Lewis      |Frank Lewis      |franklewis249@mail.com        |(910) 692-8809|Customer 